In [120]:
import pandas as pd
from pathlib import Path
import re
import numpy as np
import os
import json

h_to_eV = 27.2114
bohr_to_A = 0.529177

In [ ]:
class OrcaReader:
    """Properties from JSON file from Orca calculations
    CM5 code Modified from cclib (http://cclib.github.io) (BSD 3-Clause License)

    Enhancements:
    - Configurable .out file path (default: directory / f"{basename}.out")
    - Support multiple NBO blocks in a single output file
    - User-selectable active NBO block (default first) for NBO-derived data
    - Optional selection by COMPOUND JOB number (first NBO after header)
    - Atom label mapping: user can supply custom names for atom indices
    """

    def __init__(self, directory, basename="singlepoint", out_file_path=None, nbo_block_index=0, nbo_job_number=None):
        """Initialize OrcaReader
        Args:
            directory (str|Path): Folder containing ORCA output and JSON files
            basename (str): Base name used to locate JSON / default out file
            out_file_path (str|Path|None): Explicit path to an .out file (overrides default)
            nbo_block_index (int): Which NBO block to use (0-based) if not using job selection
            nbo_job_number (int|None): COMPOUND JOB number whose subsequent NBO block to use
        """
        self.directory = Path(directory)
        self.basename = basename
        self.outfile = Path(out_file_path) if out_file_path else self.directory / f"{self.basename}.out"
        self.nbo_block_index = nbo_block_index  # 0-based selection
        self.nbo_job_number = nbo_job_number

        self.results = {}
        self.propertyjson = {}
        self.gbwjson = {}

        self.atom_list = None
        self.atom_labels_map = {}  # int index -> custom label string

        # NBO handling
        self.nbo_sections = []       # list of full text blocks
        self.nbo_section = None      # currently selected block

        # Cached outfile parsing
        self._outfile_content = None
        self._nbo_blocks_meta = []   # list of (start_idx, end_idx) for blocks
        self._job_headers = []       # list of (job_number, start_idx)

        self.nboresults = {}  # will hold parsed NBO data for current block

    # -------------------- internal clear helper --------------------
    def _clear_nbo_data(self):
        """Clear previously parsed NBO results and any NBO charge entries in self.results."""
        nbo_keys = [k for k in self.results.keys() if k.startswith("nbo_charge_")]
        for k in nbo_keys:
            del self.results[k]
        self.nboresults = {}

    # -------------------- atom label helpers --------------------
    def define_atom_list(self, atom_list):
        """Define the list of atom indices of interest (1-based).
        Existing labels remain for overlapping indices; remove labels for indices no longer present."""
        prev = set(self.atom_list or [])
        self.atom_list = atom_list
        # Drop labels for removed indices
        for idx in list(self.atom_labels_map.keys()):
            if idx not in atom_list:
                del self.atom_labels_map[idx]

    def define_atom_labels(self, labels):
        """Define custom labels for atoms.
        labels can be:
        - list of strings same length/order as current atom_list
        - dict mapping atom_index (int) -> label string
        Labels should exclude the property prefix; property functions will prepend like 'mulliken_' automatically.
        """
        if self.atom_list is None:
            print("Define atom_list before setting labels.")
            return
        if isinstance(labels, dict):
            for k,v in labels.items():
                if k in self.atom_list and isinstance(v,str):
                    self.atom_labels_map[k] = v.strip()
        elif isinstance(labels, (list, tuple)):
            if len(labels) != len(self.atom_list):
                print("Label list length must match atom_list length.")
            else:
                for idx, name in zip(self.atom_list, labels):
                    if isinstance(name,str):
                        self.atom_labels_map[idx] = name.strip()
        else:
            print("Labels must be dict or list/tuple of strings.")

    def _atom_key(self, prefix, atom_index):
        """Return result key using custom label if present else index."""
        label = self.atom_labels_map.get(atom_index)
        if label:
            return f"{prefix}_{label}"
        return f"{prefix}_{atom_index}"

    # -------------------- configuration helpers --------------------
    def set_outfile(self, out_file_path):
        changed = Path(out_file_path) != self.outfile
        self.outfile = Path(out_file_path)
        if changed:
            self.nbo_sections = []
            self.nbo_section = None
            self._outfile_content = None
            self._nbo_blocks_meta = []
            self._job_headers = []
            self._clear_nbo_data()

    def set_nbo_block_index(self, index:int):
        prev_index = self.nbo_block_index
        self.nbo_block_index = index
        if self.nbo_sections:
            if 0 <= index < len(self.nbo_sections):
                if index != prev_index:
                    self.nbo_section = self.nbo_sections[index]
                    self._clear_nbo_data()
            else:
                print(f"Requested NBO block index {index} out of range (found {len(self.nbo_sections)} blocks).")

    def set_nbo_job_number(self, job_number:int|None):
        prev_job = self.nbo_job_number
        self.nbo_job_number = job_number
        if prev_job != job_number:
            self.nbo_section = None
            self._clear_nbo_data()

    # -------------------- generic safe traversal helpers --------------------
    @staticmethod
    def _nested_get(root, path):
        cur = root
        for p in path:
            try:
                cur = cur[p]
            except (KeyError, IndexError, TypeError):
                return None
        return cur

    def _last_geometry(self):
        geoms = self._nested_get(self.propertyjson, ["Geometries"])
        if not geoms or not isinstance(geoms, list):
            return None
        if not geoms:
            return None
        return geoms[-1]

    # -------------------- JSON parsing --------------------
    def parse_property_json(self):
        file_name = f'{self.basename}.property.json'
        if file_name in os.listdir(self.directory):
            try:
                self.propertyjson = json.load(open(self.directory / file_name))
            except Exception as e:
                print(f"Failed to load {file_name}: {e}")
                self.propertyjson = {}
        else:
            print(f"No {self.basename}.property.json file found in the specified path. Results dictionary will be empty.")

    def parse_gbw_json(self):
        file_name = f'{self.basename}.json'
        if file_name in os.listdir(self.directory):
            try:
                self.gbwjson = json.load(open(self.directory / file_name))
            except Exception as e:
                print(f"Failed to load {file_name}: {e}")
                self.gbwjson = {}
        else:
            print(f"No {self.basename}.json file found in the specified path. Results dictionary will be empty.")

    # -------------------- charge extraction (safe) --------------------
    def get_mulliken_charges(self):
        if self.atom_list is None:
            print("Atom list is not defined. Please define it using define_atom_list().")
            return
        geom = self._last_geometry()
        charges_table = self._nested_get(geom, ["Mulliken_Population_Analysis", 0, "AtomicCharges"]) if geom else None
        if charges_table is None:
            print("Mulliken charges not found; skipping.")
            return
        for at_idx in self.atom_list:
            value = None
            entry = self._nested_get(charges_table, [at_idx-1])
            if entry and isinstance(entry, list) and entry:
                value = entry[0]
            self.results[self._atom_key("mulliken", at_idx)] = value

    def get_loewdin_charges(self):
        if self.atom_list is None:
            print("Atom list is not defined. Please define it using define_atom_list().")
            return
        geom = self._last_geometry()
        charges_table = self._nested_get(geom, ["Loewdin_Population_Analysis", 0, "AtomicCharges"]) if geom else None
        if charges_table is None:
            print("Loewdin charges not found; skipping.")
            return
        for at_idx in self.atom_list:
            value = None
            entry = self._nested_get(charges_table, [at_idx-1])
            if entry and isinstance(entry, list) and entry:
                value = entry[0]
            self.results[self._atom_key("loewdin", at_idx)] = value

    def get_chelpg_charges(self):
        if self.atom_list is None:
            print("Atom list is not defined. Please define it using define_atom_list().")
            return
        geom = self._last_geometry()
        charges_table = self._nested_get(geom, ["CHELPG_Population_Analysis", 0, "AtomicCharges"]) if geom else None
        if charges_table is None:
            print("CHELPG charges not found; skipping.")
            return
        for at_idx in self.atom_list:
            value = None
            entry = self._nested_get(charges_table, [at_idx-1])
            if entry and isinstance(entry, list) and entry:
                value = entry[0]
            # Preserve original capitalization pattern chelpG
            self.results[self._atom_key("chelpG", at_idx)] = value

    def get_hirshfeld_charges(self):
        if self.atom_list is None:
            print("Atom list is not defined. Please define it using define_atom_list().")
            return
        geom = self._last_geometry()
        charges_table = self._nested_get(geom, ["Hirshfeld_Population_Analysis", 0, "AtomicCharges"]) if geom else None
        if charges_table is None:
            print("Hirshfeld charges not found; skipping.")
            return
        for at_idx in self.atom_list:
            value = None
            entry = self._nested_get(charges_table, [at_idx-1])
            if entry and isinstance(entry, list) and entry:
                value = entry[0]
            self.results[self._atom_key("hirshfeld", at_idx)] = value

    def _radii_cordero_pyykko(self):
        atomradius = np.zeros(119)
        atomradius[0] = 0.20
        for line in covalent_radius.CorderoPyykko.split("\n"):
            if not line or line.startswith('#'):
                continue
            fields = line.split()
            Z = int(fields[0])
            while len(fields) < 6:
                fields.append("0.0")
            if fields[2] == "-":
                rC = 0.0
            else:
                rC = float(fields[2].split("(")[0])
            r1 = float(fields[3])
            r1_avg = (rC + r1) / 2 if r1 != 0.0 else rC
            if getattr(self, 'atomradius_setting', 'CorderoPyykko') == "CorderoPyykko":
                atomradius[Z] = r1_avg
            elif self.atomradius_setting == "Cordero":
                atomradius[Z] = rC
        return atomradius

    def _radii_hokru(self):
        atomradius = np.array([
            0.0,
            0.32,0.37,
            1.30,0.99,0.84,0.75,0.71,0.64,0.60,0.62,
            1.60,1.40,1.24,1.14,1.09,1.04,1.00,1.01,
            2.00,1.74,1.59,1.48,1.44,1.30,1.29,1.24,1.18,1.17,1.22,1.20,1.23,1.20,1.20,1.18,1.17,1.24,
            2.15,1.90,1.78,1.64,1.56,1.46,1.38,1.36,1.34,1.30,1.36,1.40,1.42,1.40,1.40,1.37,1.32,1.36,
            2.38,2.06,
            1.94,1.84,1.90,1.73,1.86,1.85,1.83,1.82,1.81,1.80,1.79,1.77,1.77,1.78,1.74,
            1.64,1.58,1.50,1.41,1.36,1.32,1.30,1.64,1.88,1.48,1.45,1.50,1.42,1.47,1.46,
            2.42,2.11,2.01,1.90,1.84,1.83,1.80,1.80
        ])
        return atomradius

    @staticmethod
    def _tij(i: int, j: int, extended: bool = True) -> float:
        tij = 0.0
        dzz = np.array([0.0, 0.0502, 0.1747, 0.1671, 0.0556, 0.0234, -0.0346])
        if i == 1:
            if j == 6: tij = dzz[1]
            elif j == 7: tij = dzz[2]
            elif j == 8: tij = dzz[3]
        elif i == 6:
            if j == 7: tij = dzz[4]
            elif j == 8: tij = dzz[5]
            elif j == 1: tij = -dzz[1]
        elif i == 7:
            if j == 8: tij = dzz[6]
            elif j == 1: tij = -dzz[2]
            elif j == 6: tij = -dzz[4]
        elif i == 8:
            if j == 1: tij = -dzz[3]
            elif j == 6: tij = -dzz[5]
            elif j == 7: tij = -dzz[6]
        else:
            dz = np.zeros((119,))
            dz[1] = 0.0056; dz[2] = -0.1543; dz[4] = 0.0333; dz[5] = -0.1030; dz[6] = -0.0446; dz[7] = -0.1072
            dz[8] = -0.0802; dz[9] = -0.0629; dz[10] = -0.1088; dz[11] = 0.0184; dz[13] = -0.0726; dz[14] = -0.0790
            dz[15] = -0.0756; dz[16] = -0.0565; dz[17] = -0.0444; dz[18] = -0.0767; dz[19] = 0.0130
            if extended: dz[31] = -0.0512
            dz[32] = -0.0557; dz[33] = -0.0533; dz[34] = -0.0399; dz[35] = -0.0313
            if extended:
                dz[36] = -0.0541; dz[37] = 0.0092; dz[49] = -0.0361; dz[50] = -0.0393; dz[51] = -0.0376; dz[52] = -0.0281
            dz[53] = -0.0220
            if extended:
                dz[54] = -0.0381; dz[55] = 0.0065; dz[81] = -0.0255; dz[82] = -0.0277; dz[83] = -0.0265; dz[84] = -0.0198
                dz[85] = -0.0155; dz[86] = -0.0269; dz[87] = 0.0046; dz[113] = -0.0179; dz[114] = -0.0195; dz[115] = -0.0187
                dz[116] = -0.0140; dz[117] = -0.0110; dz[118] = -0.0189
            tij = dz[i] - dz[j]
        return tij

    def get_cm5_charges(self, extended: bool = True, atomradius_setting = "hokru"):
        if self.atom_list is None:
            print("Atom list is not defined. Please define it using define_atom_list().")
            return
        geom = self._last_geometry()
        if geom is None:
            print("No geometry data found; skipping CM5 charges.")
            return
        hir_raw = self._nested_get(geom, ["Hirshfeld_Population_Analysis", 0, "AtomicCharges"])
        atnos = self._nested_get(geom, ["Hirshfeld_Population_Analysis", 0, "ATNO"])
        cart = self._nested_get(geom, ["Geometry", "Coordinates", "Cartesians"])
        if hir_raw is None or atnos is None or cart is None:
            print("Missing Hirshfeld / ATNO / Cartesian data; skipping CM5 charges.")
            return
        try:
            hirshfeld_charges = np.array([x[0] for x in hir_raw])
            z = np.array([x[0] for x in atnos])
            xyz = np.array([[x,y,zv] for _, x, y, zv in cart]) * bohr_to_A
        except Exception as e:
            print(f"Failed preparing CM5 inputs: {e}")
            return
        self.atomradius_setting = atomradius_setting
        if atomradius_setting in ("CorderoPyykko", "Cordero"):
            self.atomradius = self._radii_cordero_pyykko()
        else:
            self.atomradius = self._radii_hokru()
        alpha = 2.474
        for atom_number in self.atom_list:
            i = atom_number - 1
            if i >= len(hirshfeld_charges):
                self.results[self._atom_key("cm5", atom_number)] = None
                continue
            s = 0.0
            for j in range(len(hirshfeld_charges)):
                if i == j: continue
                try:
                    rij = np.linalg.norm(xyz[i] - xyz[j])
                except Exception:
                    continue
                bij = np.exp(-alpha * (rij - self.atomradius[z[i]] - self.atomradius[z[j]]))
                tij = self._tij(z[i], z[j], extended=extended)
                s += tij * bij
            self.results[self._atom_key("cm5", atom_number)] = hirshfeld_charges[i] + s

    def get_all_charges(self):
        self.get_mulliken_charges()
        self.get_loewdin_charges()
        self.get_chelpg_charges()
        self.get_hirshfeld_charges()
        self.get_cm5_charges()
        self.get_nbo_charges()

    def get_scfenergy(self):
        geom = self._last_geometry()
        value = self._nested_get(geom, ["DFT_Energy", "finalEn"]) if geom else None
        if value is None:
            print("SCF energy not found.")
        self.results["SCF_energy"] = value

    def get_homolumo(self):
        mos = self._nested_get(self.gbwjson, ["Molecule", "MolecularOrbitals", "MOs"])
        if mos is None:
            print("MO data not found; skipping HOMO/LUMO.")
            self.results["HOMO_energy"] = None
            self.results["LUMO_energy"] = None
            return
        mo_energies = []
        mo_occupancy = []
        for mo in mos:
            try:
                mo_energies.append(mo.get("OrbitalEnergy"))
                mo_occupancy.append(mo.get("Occupancy"))
            except Exception:
                continue
        mo_energies = np.array([e for e in mo_energies if e is not None])
        mo_occupancy = np.array([o for o in mo_occupancy if o is not None])
        if mo_energies.size == 0 or mo_occupancy.size == 0:
            print("Incomplete MO data; skipping HOMO/LUMO.")
            self.results["HOMO_energy"] = None
            self.results["LUMO_energy"] = None
            return
        try:
            homo_mask = (mo_occupancy == 1.0) | (mo_occupancy == 2.0)
            HOMO_energy = np.max(mo_energies[homo_mask]) * h_to_eV if np.any(homo_mask) else None
            lumo_mask = (mo_occupancy == 0.0)
            LUMO_energy = np.min(mo_energies[lumo_mask]) * h_to_eV if np.any(lumo_mask) else None
        except Exception:
            HOMO_energy = None
            LUMO_energy = None
        self.results["HOMO_energy"] = HOMO_energy
        self.results["LUMO_energy"] = LUMO_energy

    def get_enthalpy(self):
        geom = self._last_geometry()
        value = self._nested_get(geom, ["THERMOCHEMISTRY_Energies", "Enthalpy"]) if geom else None
        if value is None:
            print("Enthalpy not found.")
        self.results["Enthalpy"] = value

    def get_saniso(self):
        if self.atom_list is None:
            print("Atom list is not defined. Please define it using define_atom_list().")
            return
        geom = self._last_geometry()
        table = self._nested_get(geom, ["Chemical_Shift", 0, "saniso"]) if geom else None
        if table is None:
            print("saniso data not found; skipping.")
            return
        for at_idx in self.atom_list:
            value = table[at_idx-1] if at_idx-1 < len(table) else None
            self.results[self._atom_key("saniso", at_idx)] = value

    def get_siso(self):
        if self.atom_list is None:
            print("Atom list is not defined. Please define it using define_atom_list().")
            return
        geom = self._last_geometry()
        table = self._nested_get(geom, ["Chemical_Shift", 0, "siso"]) if geom else None
        if table is None:
            print("siso data not found; skipping.")
            return
        for at_idx in self.atom_list:
            value = table[at_idx-1] if at_idx-1 < len(table) else None
            self.results[self._atom_key("siso", at_idx)] = value

    def _ensure_outfile_content(self):
        if self._outfile_content is None:
            try:
                self._outfile_content = self.outfile.read_text()
            except Exception as e:
                print(f"Failed to read output file '{self.outfile}': {e}")
                self._outfile_content = ""

    def _scan_job_headers(self):
        self._ensure_outfile_content()
        text = self._outfile_content
        self._job_headers = []
        if not text:
            return
        job_re = re.compile(r"(?m)^[ \t]*COMPOUND[ \t]+JOB[ \t]+(?:\{[ \t]*(\d+)[ \t]*\}|(\d+))[ \t]*$")
        for m in job_re.finditer(text):
            num = m.group(1) or m.group(2)
            try:
                num_i = int(num)
            except Exception:
                continue
            self._job_headers.append((num_i, m.start()))
        self._job_headers.sort(key=lambda x: x[1])

    def _scan_nbo_blocks(self):
        self._ensure_outfile_content()
        text = self._outfile_content
        self.nbo_sections = []
        self._nbo_blocks_meta = []
        if not text:
            return
        start_pattern = "Now starting NBO"
        end_pattern = "*** returned from  NBO  program ***"
        start_positions = []
        idx = 0
        while True:
            pos = text.find(start_pattern, idx)
            if pos == -1:
                break
            start_positions.append(pos)
            idx = pos + len(start_pattern)
        if not start_positions:
            print("No NBO start markers found in output file.")
            return
        for s in start_positions:
            e = text.find(end_pattern, s)
            if e == -1:
                print(f"Warning: NBO start at {s} without matching end marker; skipping.")
                continue
            block = text[s:e + len(end_pattern)]
            self.nbo_sections.append(block)
            self._nbo_blocks_meta.append((s, e + len(end_pattern)))
        if not self.nbo_sections:
            print("No complete NBO blocks detected.")

    def _select_nbo_block_by_job(self, job_number:int):
        if not self._job_headers:
            self._scan_job_headers()
        if not self._job_headers:
            print("No COMPOUND JOB headers found; falling back to first NBO block.")
            return 0 if self.nbo_sections else None
        header_pos = None
        next_header_pos = None
        for i, (num, pos) in enumerate(self._job_headers):
            if num == job_number:
                header_pos = pos
                if i + 1 < len(self._job_headers):
                    next_header_pos = self._job_headers[i+1][1]
                break
        if header_pos is None:
            print(f"COMPOUND JOB {job_number} not found; falling back to first NBO block.")
            return 0 if self.nbo_sections else None
        for idx, (s, e) in enumerate(self._nbo_blocks_meta):
            if s >= header_pos and (next_header_pos is None or s < next_header_pos):
                return idx
        print(f"No NBO block found after COMPOUND JOB {job_number}; falling back to first NBO block.")
        return 0 if self.nbo_sections else None

    def init_nbo(self, job_number:int|None=None):
        self._clear_nbo_data()
        if not self.nbo_sections or not self._nbo_blocks_meta:
            self._scan_nbo_blocks()
        selected_index = None
        effective_job = job_number if job_number is not None else self.nbo_job_number
        if effective_job is not None:
            selected_index = self._select_nbo_block_by_job(effective_job)
        if selected_index is None:
            if not self.nbo_sections:
                self.nbo_section = None
                return
            if not (0 <= self.nbo_block_index < len(self.nbo_sections)):
                print(f"Block index {self.nbo_block_index} out of range; defaulting to 0.")
                self.nbo_block_index = 0
            selected_index = self.nbo_block_index
        self.nbo_block_index = selected_index
        self.nbo_section = self.nbo_sections[selected_index]
        self.nboresults = {
            'source_file': str(self.outfile),
            'nbo_block_index': self.nbo_block_index,
            'job_number': effective_job,
            'analysis_type': 'NBO 7.0',
            'natural_population_analysis': {'atoms': {}, 'totals': {}},
            'natural_atomic_orbitals': {},
            'natural_bond_orbitals': {},
            'bond_orbital_details': {},
            'nho_directionality': {},
            'second_order_perturbation': {},
            'analysis_info': {}
        }

    def get_NPA(self):
        if self.nbo_section is None:
            self.init_nbo()
        if self.nbo_section is None:
            return
        npa_start = self.nbo_section.find("Summary of Natural Population Analysis:")
        npa_end = self.nbo_section.find("Natural Electron Configuration", npa_start)
        if npa_end == -1:
            npa_end = self.nbo_section.find("\n\n NATURAL BOND ORBITAL ANALYSIS:", npa_start)
        if npa_start != -1 and npa_end != -1:
            npa_section = self.nbo_section[npa_start:npa_end]
            atom_properties = []
            atom_lines = re.findall(r'\n\s+([A-Z][a-z]?)\s+(\d+)\s+([-\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)', npa_section)
            for atom_data in atom_lines:
                atom_symbol = atom_data[0]
                atom_num = int(atom_data[1])
                while len(atom_properties) <= atom_num:
                    atom_properties.append(None)
                atom_properties[atom_num] = {
                    'atom': atom_symbol,
                    'charge': float(atom_data[2]),
                    'core': float(atom_data[3]),
                    'valence': float(atom_data[4]),
                    'rydberg': float(atom_data[5]),
                    'total': float(atom_data[6])
                }
            self.nboresults['natural_population_analysis']['atoms'] = {i: prop for i, prop in enumerate(atom_properties) if prop is not None}
            total_match = re.search(r'\*\s*Total\s*\*\s+([-\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)', npa_section)
            if total_match:
                total_properties = {
                    'charge': float(total_match.group(1)),
                    'core': float(total_match.group(2)),
                    'valence': float(total_match.group(3)),
                    'rydberg': float(total_match.group(4)),
                    'total': float(total_match.group(5))
                }
                self.nboresults['natural_population_analysis']['totals'] = total_properties
        else:
            print("Could not find Natural Population Analysis section")

    def get_NAO(self):
        if self.nbo_section is None:
            self.init_nbo()
        if self.nbo_section is None:
            return
        nao_start = self.nbo_section.find("NATURAL POPULATIONS:  Natural atomic orbital occupancies")
        nao_end = self.nbo_section.find("Summary of Natural Population Analysis:", nao_start)
        nao_properties = []
        if nao_start != -1 and nao_end != -1:
            nao_subsection = self.nbo_section[nao_start:nao_end]
            nao_lines = re.findall(r'\n\s+(\d+)\s+([A-Z][a-z]?)\s+(\d+)\s+(\S+)\s+(\S+\([^)]+\))\s+([\d.]+)\s+([-\d.]+)', nao_subsection)
            for line_data in nao_lines:
                orbital_type = line_data[4].split('(')[0]
                orbital = line_data[4].split('(')[1].strip(')').strip()
                nao_properties.append({
                    'orbital_num': int(line_data[0]),
                    'atom': line_data[1],
                    'atom_num': int(line_data[2]),
                    'angular': line_data[3],
                    'orbital_type': orbital_type,
                    'orbital': orbital,
                    'occupancy': float(line_data[5]),
                    'energy': float(line_data[6])
                })
            self.nboresults['natural_atomic_orbitals'] = nao_properties
        else:
            print("Could not find NAO section")

    def get_NBO_summary(self):
        if self.nbo_section is None:
            self.init_nbo()
        if self.nbo_section is None:
            return
        nbo_summary_start = self.nbo_section.find("NATURAL BOND ORBITALS (Summary):")
        nbo_summary_end = self.nbo_section.find("$CHOOSE", nbo_summary_start)
        if nbo_summary_end == -1:
            nbo_summary_end = self.nbo_section.find("Checkpoint orbital", nbo_summary_start)
        nbo_properties = []
        if nbo_summary_start != -1 and nbo_summary_end != -1:
            nbo_summary_section = self.nbo_section[nbo_summary_start:nbo_summary_end]
            nbo_lines = re.findall(r'\n\s+(\d+)\.\s+(BD\*?|LP|CR|RY)\s*\(\s*(\d+)\)\s+([A-Z][a-z]?\s+\d+(?:\s*-\s*[A-Z][a-z]?\s+\d+)*)\s+([\d.]+)\s+([-\d.]+)', nbo_summary_section)
            for line_data in nbo_lines:
                atoms_string = line_data[3].strip()
                atom_numbers = [int(m[1]) for m in re.findall(r'([A-Z][a-z]?)\s+(\d+)', atoms_string)]
                nbo_properties.append({
                    'nbo_num': int(line_data[0]),
                    'nbo_type': line_data[1],
                    'bond_index': int(line_data[2]),
                    'atoms': atom_numbers,
                    'occupancy': float(line_data[4]),
                    'energy': float(line_data[5])
                })
            self.nboresults['natural_bond_orbitals'] = nbo_properties
        else:
            print("Could not find NBO summary section")

    def get_orbital_details(self):
        if self.nbo_section is None:
            self.init_nbo()
        if self.nbo_section is None:
            return
        bond_detail_start = self.nbo_section.find("(Occupancy)   Bond orbital / Coefficients / Hybrids")
        lewis_start = self.nbo_section.find("---------------- Lewis", bond_detail_start)
        non_lewis_start = self.nbo_section.find("---------------- non-Lewis", lewis_start)
        bond_orbital_details = []
        if bond_detail_start != -1 and lewis_start != -1:
            if non_lewis_start != -1:
                end_markers = ["NHO DIRECTIONALITY", "SECOND ORDER", "\n\n NATURAL BOND ORBITALS (Summary)"]
                detail_end = -1
                for marker in end_markers:
                    pos = self.nbo_section.find(marker, non_lewis_start)
                    if pos != -1:
                        detail_end = pos; break
            else:
                detail_end = -1
                end_markers = ["NHO DIRECTIONALITY", "SECOND ORDER", "\n\n NATURAL BOND ORBITALS (Summary)"]
                for marker in end_markers:
                    pos = self.nbo_section.find(marker, lewis_start)
                    if pos != -1:
                        detail_end = pos; break
            if detail_end != -1:
                bond_detail_section = self.nbo_section[lewis_start:detail_end]
                bond_entries = re.finditer(r'\n\s+(\d+)\.\s+\(([\d.]+)\)\s+(BD\*?|LP|CR|RY)\s*\(\s*(\d+)\)\s+(.*?)(?=\n\s+\d+\.\s+\(|\Z)', bond_detail_section, re.DOTALL)
                for bond_entry in bond_entries:
                    bond_num = int(bond_entry.group(1))
                    occupancy = float(bond_entry.group(2))
                    bond_type = bond_entry.group(3)
                    bond_index = int(bond_entry.group(4))
                    atoms_and_contrib = bond_entry.group(5)
                    atoms_line_match = re.match(r'(.*?)(?=\n|\()', atoms_and_contrib)
                    atoms_line = atoms_line_match.group(1).strip() if atoms_line_match else ""
                    atom_numbers = [int(m[1]) for m in re.findall(r'([A-Z][a-z]?)\s+(\d+)', atoms_line)]
                    bond_info = {
                        'bond_num': bond_num,
                        'occupancy': occupancy,
                        'bond_type': bond_type,
                        'bond_index': bond_index,
                        'atoms': atom_numbers,
                        'contributions': []
                    }
                    contrib_pattern = r'\(\s*([\d.]+)%\)\s+([-\d.]+)\*\s+([A-Z][a-z]?)\s+(\d+)\s+s\(\s*([\d.]+)%\)p\s*([\d.]+)\(\s*([\d.]+)%\)'
                    contributions = re.finditer(contrib_pattern, atoms_and_contrib)
                    for contrib in contributions:
                        bond_info['contributions'].append({
                            'percentage': float(contrib.group(1)),
                            'coefficient': float(contrib.group(2)),
                            'atom': contrib.group(3),
                            'atom_num': int(contrib.group(4)),
                            's_percent': float(contrib.group(5)),
                            'p_power': float(contrib.group(6)),
                            'p_percent': float(contrib.group(7))
                        })
                    bond_orbital_details.append(bond_info)
                self.nboresults['bond_orbital_details'] = bond_orbital_details
            else:
                print("Could not determine end of bond detail section")
        else:
            print("Could not find bond orbital detail section")

    def get_NHO_directionality(self):
        if self.nbo_section is None:
            self.init_nbo()
        if self.nbo_section is None:
            return
        dir_start = self.nbo_section.find("NHO DIRECTIONALITY AND BOND BENDING")
        dir_end = self.nbo_section.find("\n SECOND ORDER PERTURBATION", dir_start)
        if dir_end == -1: dir_end = self.nbo_section.find("\n\n NATURAL BOND ORBITALS", dir_start)
        nbo_directionality = []
        if dir_start != -1 and dir_end != -1:
            dir_section = self.nbo_section[dir_start:dir_end]
            if "None exceeding thresholds" not in dir_section:
                dir_lines = re.findall(r'\n\s+(\d+)\.\s+(BD|LP|CR)\s*\(\s*(\d+)\)\s+([A-Z][a-z]?\s+\d+(?:\s*-\s*[A-Z][a-z]?\s+\d+)*)\s+([\d.-]+)\s+([\d.-]+)\s+([\d.-]+)\s+([\d.-]+)\s+([\d.-]+)', dir_section)
                for line_data in dir_lines:
                    atoms_string = line_data[3].strip()
                    atom_numbers = [int(m[1]) for m in re.findall(r'([A-Z][a-z]?)\s+(\d+)', atoms_string)]
                    nbo_directionality.append({
                        'nbo_num': int(line_data[0]),
                        'nbo_type': line_data[1],
                        'bond_index': int(line_data[2]),
                        'atoms': atom_numbers,
                        'line_theta': float(line_data[4]) if line_data[4] != '--' else None,
                        'line_phi': float(line_data[5]) if line_data[5] != '--' else None,
                        'hybrid_theta': float(line_data[6]) if line_data[6] != '--' else None,
                        'hybrid_phi': float(line_data[7]) if line_data[7] != '--' else None,
                        'deviation': float(line_data[8]) if line_data[8] != '--' else None
                    })
            self.nboresults['nho_directionality'] = nbo_directionality
        else:
            print("Could not find NHO Directionality section")

    def get_second_order_perturbation(self):
        if self.nbo_section is None:
            self.init_nbo()
        if self.nbo_section is None:
            return
        second_start = self.nbo_section.find("SECOND ORDER PERTURBATION THEORY ANALYSIS")
        second_end = self.nbo_section.find("\n\n NATURAL BOND ORBITALS (Summary):", second_start)
        if second_end == -1: second_end = self.nbo_section.find("\n $CHOOSE", second_start)
        second_order_perturbation = []
        if second_start != -1 and second_end != -1:
            second_section = self.nbo_section[second_start:second_end]
            if "None above threshold" not in second_section and "within unit" in second_section:
                interaction_pattern = r'\n\s+(BD\*?|LP|CR|RY)\s*\(\s*(\d+)\)\s+(.*?)\s+->\s+(BD\*?|LP\*?|CR\*?|RY)\s*\(\s*(\d+)\)\s+(.*?)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)'
                interactions = re.finditer(interaction_pattern, second_section)
                for inter in interactions:
                    second_order_perturbation.append({
                        'donor_type': inter.group(1),
                        'donor_index': int(inter.group(2)),
                        'donor_atoms': inter.group(3).strip(),
                        'acceptor_type': inter.group(4),
                        'acceptor_index': int(inter.group(5)),
                        'acceptor_atoms': inter.group(6).strip(),
                        'e2_energy': float(inter.group(7)),
                        'energy_diff': float(inter.group(8)),
                        'fock_element': float(inter.group(9))
                    })
            self.nboresults['second_order_perturbation'] = second_order_perturbation
        else:
            print("Could not find Second-Order Perturbation section")

    def get_cycle_info(self):
        if self.nbo_section is None:
            self.init_nbo()
        if self.nbo_section is None:
            return
        cycle_start = self.nbo_section.find("NATURAL BOND ORBITAL ANALYSIS:")
        cycle_end = self.nbo_section.find("Structure accepted:", cycle_start)
        if cycle_end == -1:
            cycle_end = self.nbo_section.find("\n\n", cycle_start + 100)
        nbo_analysis_info = {}
        if cycle_start != -1 and cycle_end != -1:
            cycle_section = self.nbo_section[cycle_start:cycle_end + 200]
            cycle_match = re.search(r'\n\s+(\d+)\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)', cycle_section)
            if cycle_match:
                nbo_analysis_info = {
                    'cycle': int(cycle_match.group(1)),
                    'center': int(cycle_match.group(2)),
                    'threshold': float(cycle_match.group(3)),
                    'lewis_occupancy': float(cycle_match.group(4)),
                    'non_lewis_occupancy': float(cycle_match.group(5)),
                    'num_core': int(cycle_match.group(6)),
                    'num_bonds': int(cycle_match.group(7)),
                    'num_non_bonding_cores': int(cycle_match.group(8)),
                    'num_lone_pairs': int(cycle_match.group(9)),
                    'low_occ_lewis': int(cycle_match.group(10)),
                    'high_occ_non_lewis': int(cycle_match.group(11))
                }
                self.nboresults['analysis_info'] = nbo_analysis_info
        else:
            print("Could not find NBO analysis section")

    def get_nbo_charges(self):
        if self.atom_list is None:
            print("Atom list is not defined. Please define it using define_atom_list().")
            return
        if self.nbo_section is None:
            self.init_nbo()
        if self.nbo_section is None:
            return
        if not self.nboresults.get('natural_population_analysis', {}).get('atoms'):
            self.get_NPA()
        atoms_dict = self.nboresults.get('natural_population_analysis', {}).get('atoms', {})
        for at_idx in self.atom_list:
            npa_data = atoms_dict.get(at_idx)
            self.results[self._atom_key("nbo_charge", at_idx)] = npa_data['charge'] if npa_data else None

    def write_results_to_file(self, output_path=None):
        if output_path is None:
            output_path = self.directory / f"{self.basename}_descriptors.json"
        try:
            with open(output_path, 'w') as f:
                json.dump(self.results, f, indent=4)
        except Exception as e:
            print(f"Failed writing results file: {e}")

In [131]:
class RedoxSet:
    def __init__(self, neutralReader=None, cationReader=None, anionReader=None, descriptorPrefixes=None):
        self.neutralReader = neutralReader
        self.cationReader = cationReader
        self.anionReader = anionReader
        self.descriptorPrefixes = descriptorPrefixes or ["mulliken", "loewdin", "nbo_charge", "cm5", "chelpg", "hirshfeld"]
        self.results = {}
        # Adopt atom list and labels from neutral reader if available
        self.atom_labels_map = {}
        self.atom_list = None
        if self.neutralReader and getattr(self.neutralReader, 'atom_list', None):
            self.atom_list = list(self.neutralReader.atom_list)
            self.atom_labels_map = dict(getattr(self.neutralReader, 'atom_labels_map', {}))

    def _atom_key(self, prefix, atom_index):
        label = self.atom_labels_map.get(atom_index)
        if label:
            return f"{prefix}_{label}"
        return f"{prefix}_{atom_index}"

    def _pull_charge(self, reader, prefix, atom_index):
        if reader is None:
            return None
        # Attempt labeled key first
        labeled = self._atom_key(prefix, atom_index)
        if labeled in reader.results:
            return reader.results.get(labeled)
        # Fallback to numeric key
        return reader.results.get(f"{prefix}_{atom_index}")

    def get_redox_properties(self):
        self.results['Vertical_Ionization_Potential'] = self.neutralReader.results.get('SCF_energy') - self.cationReader.results.get('SCF_energy') if (self.neutralReader and self.cationReader and self.neutralReader.results.get('SCF_energy') is not None and self.cationReader.results.get('SCF_energy') is not None) else None
        self.results['Vertical_Electron_Affinity'] = self.anionReader.results.get('SCF_energy') - self.neutralReader.results.get('SCF_energy') if (self.neutralReader and self.anionReader and self.anionReader.results.get('SCF_energy') is not None and self.neutralReader.results.get('SCF_energy') is not None) else None
        ip = self.results['Vertical_Ionization_Potential']
        ea = self.results['Vertical_Electron_Affinity']
        if ip is not None and ea is not None:
            hardness = (ip - ea) / 2
            electronegativity = (ip + ea) / 2
            self.results['Hardness'] = hardness
            self.results['Electronegativity'] = electronegativity
            self.results['Electrophilicity'] = (electronegativity ** 2) / (2 * hardness) if hardness else None
        else:
            self.results['Hardness'] = None
            self.results['Electronegativity'] = None
            self.results['Electrophilicity'] = None

    def get_fukui_functions(self):
        if not self.atom_list:
            print("Atom list not available in RedoxSet.")
            return
        for prefix in self.descriptorPrefixes:
            for idx in self.atom_list:
                qN = self._pull_charge(self.neutralReader, prefix, idx)
                qC = self._pull_charge(self.cationReader, prefix, idx)
                qA = self._pull_charge(self.anionReader, prefix, idx)
                f_plus = (qN - qA) if (qN is not None and qA is not None) else None
                f_minus = (qC - qN) if (qC is not None and qN is not None) else None
                f_zero = (0.5 * (f_plus + f_minus)) if (f_plus is not None and f_minus is not None) else None
                self.results[self._atom_key(f"fplus_{prefix}", idx)] = f_plus
                self.results[self._atom_key(f"fminus_{prefix}", idx)] = f_minus
                self.results[self._atom_key(f"fzero_{prefix}", idx)] = f_zero

    def aggregate(self):
        self.get_redox_properties()
        self.get_fukui_functions()
        return self.results

In [123]:
test = OrcaReader(directory="orca_exampleNi_2",basename="singlepoint", out_file_path="orca_exampleNi_2/calcParam_initialOpt.out")

In [124]:
test.define_atom_list([1,2,3])
test.define_atom_labels(["Ni1","Ni2","S1"])

In [125]:
test.parse_property_json()

In [126]:
test.get_all_charges()
test.results

{'mulliken_Ni1': -0.09933822616022336,
 'mulliken_Ni2': -0.32542246849194356,
 'mulliken_S1': -0.10300224960161142,
 'loewdin_Ni1': -0.08070168626157681,
 'loewdin_Ni2': -0.3292145112127649,
 'loewdin_S1': -0.08434993632044008,
 'chelpG_Ni1': -0.15110950799203424,
 'chelpG_Ni2': -0.34395086182289114,
 'chelpG_S1': -0.1685762063735124,
 'hirshfeld_Ni1': -0.14326363077598225,
 'hirshfeld_Ni2': -0.17971358708224372,
 'hirshfeld_S1': -0.1480743828735407,
 'cm5_Ni1': np.float64(-0.13929646742574928),
 'cm5_Ni2': np.float64(-0.006035378611593117),
 'cm5_S1': np.float64(-0.1439067218259235),
 'nbo_charge_Ni1': -0.24435,
 'nbo_charge_Ni2': 0.38232,
 'nbo_charge_S1': -0.25807}

In [127]:
test.set_nbo_job_number(5)

In [128]:
test.get_all_charges()
test.results

{'mulliken_Ni1': -0.09933822616022336,
 'mulliken_Ni2': -0.32542246849194356,
 'mulliken_S1': -0.10300224960161142,
 'loewdin_Ni1': -0.08070168626157681,
 'loewdin_Ni2': -0.3292145112127649,
 'loewdin_S1': -0.08434993632044008,
 'chelpG_Ni1': -0.15110950799203424,
 'chelpG_Ni2': -0.34395086182289114,
 'chelpG_S1': -0.1685762063735124,
 'hirshfeld_Ni1': -0.14326363077598225,
 'hirshfeld_Ni2': -0.17971358708224372,
 'hirshfeld_S1': -0.1480743828735407,
 'cm5_Ni1': np.float64(-0.13929646742574928),
 'cm5_Ni2': np.float64(-0.006035378611593117),
 'cm5_S1': np.float64(-0.1439067218259235),
 'nbo_charge_Ni1': -0.11213,
 'nbo_charge_Ni2': 1.00336,
 'nbo_charge_S1': -0.13441}

In [146]:
neutral = OrcaReader(directory="orca_exampleNi_2",basename="singlepoint", out_file_path="orca_exampleNi_2/calcParam_initialOpt.out")
neutral.define_atom_list([1,2,3])
neutral.parse_property_json()
neutral.get_all_charges()
neutral.get_scfenergy()
positive = OrcaReader(directory="orca_exampleNi_2",basename="oxidation", out_file_path="orca_exampleNi_2/calcParam_initialOpt.out", nbo_job_number=5)
positive.parse_property_json()
positive.define_atom_list([1,2,3])
positive.get_all_charges()
positive.get_scfenergy()
negative = OrcaReader(directory="orca_exampleNi_2",basename="reduction", out_file_path="orca_exampleNi_2/calcParam_initialOpt.out", nbo_job_number=6)
negative.parse_property_json()
negative.define_atom_list([1,2,3])
negative.get_all_charges()
negative.get_scfenergy()
redox = RedoxSet(neutralReader=neutral, cationReader=positive, anionReader=negative)

In [147]:
redox.aggregate()

{'Vertical_Ionization_Potential': -0.2131434198192892,
 'Vertical_Electron_Affinity': -0.026379117940678043,
 'Hardness': -0.09338215093930557,
 'Electronegativity': -0.11976126887998362,
 'Electrophilicity': -0.07679605459648235,
 'fplus_mulliken_1': 0.07435182303398125,
 'fminus_mulliken_1': 0.1822177098490999,
 'fzero_mulliken_1': 0.12828476644154058,
 'fplus_mulliken_2': 0.059541898082414946,
 'fminus_mulliken_2': 0.28211992832234856,
 'fzero_mulliken_2': 0.17083091320238175,
 'fplus_mulliken_3': 0.056326149086924904,
 'fminus_mulliken_3': 0.1740559881646524,
 'fzero_mulliken_3': 0.11519106862578865,
 'fplus_loewdin_1': 0.03900607282448898,
 'fminus_loewdin_1': 0.10446639352800213,
 'fzero_loewdin_1': 0.07173623317624556,
 'fplus_loewdin_2': 0.05835606754671119,
 'fminus_loewdin_2': 0.5348883565505247,
 'fzero_loewdin_2': 0.29662221204861794,
 'fplus_loewdin_3': 0.03006297805425273,
 'fminus_loewdin_3': 0.09958439561984789,
 'fzero_loewdin_3': 0.06482368683705031,
 'fplus_nbo_charg